# パラメータ設定

---

OpenHPC環境を構築するのに必要となるパラメータを設定します。

## 概要

VCP SDKを用いてクラウド上に仮想サーバを作成し、OpenHPC環境の構築を行います。

![構成](images/ohpc-000.png)

このNotebookでは以下に示すパラメータを設定します。

* VCP SDKに関するパラメータ
* VCノードに共通するパラメータ
* 割り当てリソースに関するパラメータ
    - 計算ノード
    - マスターノード
    - NFS用ディスク
* VCノードのホスト名とIPアドレス
* Slurmに関するパラメータ

## mdx環境での利用について

mdx上でOpenHPC環境を作成する場合に必要となる前提条件と準備作業について示します。
mdx以外の環境で構築する場合は、この章をスキップしてください。

### 前提条件

* VCコントローラは、[handson202209-vcp](https://github.com/nii-gakunin-cloud/handson/tree/master/Basic-Tutorials/handson202209-vcp)に従ってmdx上に構築され、そのVCコントローラからmdx上にVCノードの作成を確認済みであること。
* このNotebookの実行環境は、VCコントローラと同一ホスト上で稼動するJupyterNotebookサーバであること。

### VcpSDK mdxプラグインの修正

VMデプロイが非同期に実行できるよう、mdxプラグインを修正します。

以下のセルを実行して、**mdxプラグインモジュールのロード前に**mdxプラグインにパッチを適用します。パッチが適用済みの場合に、パッチが適用済みか検査する段階で`2 out of 2 hunks FAILED`のように表示されますが、セルの実行が成功となれば問題はありません。

In [ ]:
import os
sdkdirs = [os.path.expanduser('~/vcp/vcpsdk'),
           os.path.expanduser('~/vcpsdk')]
mdxpatch = 'patches/mdx-deploy-wait_for-False.patch'

for d in sdkdirs:
    if os.path.exists(d):
        !if cat {mdxpatch} | (cd {d}; patch -sf -p1 --dry-run); then \
            cat {mdxpatch} | (cd {d}; patch -p1); \
        else \
            if cat {mdxpatch} | (cd {d}; patch -R -sf -p1 --dry-run); then \
                echo "Patch already applied in {d}."; \
            else \
                echo "Patch could not be applied in {d}."; \
                exit 1; \
            fi \
        fi

### jupyter-autotimeのセットアップ

mdx VMのデプロイやIPアドレス変更には数分を要します。実行中のセルの経過時間を表示するために、jupyter-autotimeをインストールし有効化します。

In [ ]:
!pip install --no-deps jupyter-autotime
%load_ext autotime

## VCP SDK

VCP SDKを利用する際に必要となるパラメータを設定します。


### VCCアクセストークンの入力

![VCP SDK](images/ohpc-001.png)

VCノードを起動するにはVC Controller(VCC)にアクセスして、操作を行う必要があります。VCCにアクセスするために必要となるアクセストークンをここで入力します。

次のセルを実行すると入力枠が表示されるのでアクセストークンの値を入力してください。

> アクセストークン入力後に Enter キーを押すことで入力が完了します。

In [ ]:
from getpass import getpass
vcc_access_token = getpass()

入力されたアクセストークンが正しいことを、実際にVCCにアクセスして確認します。

In [ ]:
from vcpsdk.vcpsdk import VcpSDK
vcp = VcpSDK(vcc_access_token)

上のセルの実行結果がエラーとなり以下のようなメッセージが表示されている場合は、入力されたアクセストークンに誤りがあります。

```
config vc failed: http_status(403)
2021/XX/XX XX:XX:XX UTC: VCPAuthException: xxxxxxx:token lookup is failed: permission denied
```

エラーになった場合はこの節のセルを全て `unfreeze` してから、もう一度アクセストークンの入力を行ってください。

> `unfreeze`するにはNotebookのツールバーにある`unfreeze below in section`ボタンなどを利用してください。

#### VCCのバージョンチェック

このテンプレートはVCC 25.04以上での利用を想定しています。利用環境のVCCが条件を満たしていることをチェックします。

VCCのバージョンを確認します。

In [ ]:
 %run scripts/vcp.py
vc_controller_version(vcp) 

VCCのバージョンが25.04以上であることをチェックします。

In [ ]:
 check_vcc_version(vcp, "25.04")

### UnitGroup名の指定

このアプリケーションテンプレートで構築するOpenHPC環境に対して、名前を付けます。指定した名前はVCPのUnitGroup名となります。

VCPの構成要素は以下のようになっています。

* VCノード
  - クラウドにおける計算資源(VM/BM)
  - 例えば Amazon EC2インスタンス, Microsoft Azure VM など
* Unit
  - 同質のVCノードにより構成されている要素
  - 同じUnitに属するVCノードはCPU,メモリ等の計算資源が全て同じ設定になっている
* UnitGroup
  - 複数のUnitにより構成されている要素
  - 使用目的、ライフサイクルなどに合わせて、複数のUnitをまとめて扱うための要素  

VCP SDKで作成した他の環境と区別するために UnitGroupに名前を付けます。UnitGroup名は既存のものと異なる値を指定する必要があります。
既存のUnitGroupを確認するために一覧を表示します。

In [ ]:
vcp.df_ugroups()

この構築環境のUnitGroup名を次のセルで指定してください。UnitGroup名は**英数字のみ**の値を指定してください（先頭文字に数字は指定できない）。

> 英数字以外の文字を指定すると、環境構築時のansibleの実行にて警告文が表示される場合があります。

In [ ]:
# (例)
# ugroup_name = 'OpenHPC'

ugroup_name = 

### パラメータの保存

この章で指定したパラメータの値をファイルに保存します。

後の手順でVCノードに対する操作を、構成管理ツールの[Ansible](https://www.ansible.com/)で行います。そこで、パラメータの保存形式は `Ansible` のフォーマットに従うことにします。Ansible では `group_vars/`というディレクトリに YAML フォーマットのファイルを配置すると、そのファイルに記録されている値を変数として利用することができます。このNotebookでは `group_vars/` にあるファイルを `group_vars ファイル`と呼ぶことにします。

値の保存を行う前に、入力されたパラメータに対して簡易なチェックを行います。エラーになった場合はその後に表示される指示に従ってください。

In [ ]:
%run scripts/utils.py
check_parameters_with_keys(globals(), ["ugroup_name"], ["vcp"])

次のセルを実行すると、この章で指定したパラメータが group_vars ファイルに保存されます。

> YAMLフォーマットでファイルに値を保存するために、事前に作成した Python のスクリプト `scripts/group.py` を利用しています。

In [ ]:
%run scripts/group.py
from pathlib import Path

update_group_vars(
    ugroup_name,
    ugroup_name=ugroup_name,
)

`group_vars`ファイルの内容を確認してみます。

In [ ]:
!cat group_vars/{ugroup_name}.yml

## VCノードに共通するパラメータ

マスターノード、計算ノードに共通するパラメータを指定します。

### SSH公開鍵認証の鍵ファイルの指定

起動したVCノードにはsshでログインして操作を行います。sshでログインできるようにするために、事前に公開鍵をVCノードに登録します。

> 公開鍵は事前にNotebook環境で`ssh-keygen`コマンドなどを用いて作成するか、他環境で作成した鍵ファイルをNotebook環境にアップロードしておいてください。

VCノードに登録するSSHの公開鍵ファイルのパスを次のセルで指定してください。

In [ ]:
# (例)
# ssh_public_key_path = '~/.ssh/id_rsa.pub'

ssh_public_key_path = 

公開鍵に対応する秘密鍵のパスを次のセルで指定してください。

In [ ]:
# (例)
# ssh_private_key_path = '~/.ssh/id_rsa'

ssh_private_key_path = 

### パラメータの保存

この章で指定したパラメータの値をファイルに保存します。

値の保存を行う前に、入力されたパラメータに対して簡易なチェックを行います。エラーになった場合はその後に表示される指示に従ってください。

In [ ]:
%run scripts/utils.py
from pathlib import Path
ssh_public_key_path = str(Path(ssh_public_key_path).expanduser())
ssh_private_key_path = str(Path(ssh_private_key_path).expanduser())
check_parameters_with_keys(globals(), ["ssh_public_key_path", "ssh_private_key_path"], ["vcp"])

この章で指定したパラメータを group_vars ファイルに保存します。

In [ ]:
%run scripts/group.py
from pathlib import Path

update_group_vars(
    ugroup_name,
    ssh_public_key_path=str(Path(ssh_public_key_path).expanduser()),
    ssh_private_key_path=str(Path(ssh_private_key_path).expanduser()),
)

`group_vars`ファイルの内容を確認してみます。

In [ ]:
!cat group_vars/{ugroup_name}.yml

## マスターノード

マスターノードに関するパラメータを指定します。

![マスターノード](images/ohpc-003.png)

### マスターノードのクラウドプロバイダ

VCノードを作成するプロバイダ名を指定します。

In [ ]:
# (例)
# master_provider = 'aws'
# master_provider = 'azure'
# master_provider = 'oracle'
# master_provider = 'onpremises'   ### mdxの場合

master_provider = 

### マスターノードのflavor

マスターノードに割り当てるリソースに対応する `flavor` の値を指定します。

VCノードの`spec`に対して種々のパラメータを毎回設定するのは煩雑な作業になります。そこでVCP SDKでは典型的なパラメータセットを事前に定義しています。事前に定義したパラメータセットのことをVCP SDKでは`flavor`と呼んでいます。`spec`に設定できるパラメータはクラウドプロバイダ毎に異なるので `flavor`もプロバイダ毎に定義されています。

次のセルを実行すると`master_provider`で指定したクラウドプロバイダ名に対応する `flavor` の一覧が表示されます。

In [ ]:
vcp.df_flavors(master_provider)

上に表示された表の `flavor` の欄の値から、マスターノードとして利用するVCノードの `flavor` を選んで次のセルで指定してください。

In [ ]:
# (例)
# master_flavor = 'small'
# master_flavor = 'default'    ### mdxの場合

master_flavor = 

### マスターノードのインスタンスタイプ

`flavor`で事前に定義されている以外のインスタンスタイプを利用したい場合は次のセルのコメントを外してインスタンスタイプを指定してください。

In [ ]:
# (例)
# master_instance_type = 'm7i.xlarge'

### マスターノードのルートボリュームサイズ

マスターノード用VCノードのルートボリュームサイズ(GB)を指定します。

OpenHPCのマスターノードのルートボリュームサイズには20GB以上の値を指定してください。また NGC カタログのコンテナを利用する場合は Singularity でコンテナイメージを変換するための作業領域が必要となるため 60GB 以上の値を指定してください。

In [ ]:
# (例)
# master_root_size = 60

master_root_size = 

### マスターノードのIPアドレスとホスト名

マスターノードに割り当てるIPアドレスを指定します。

まず、VCノードに割り当て可能なIPアドレスの範囲を確認します。

mdx上での構築の場合には、以下のセルではアドレス範囲を取得できないため、[mdxでの静的IPアドレス設定方法](https://docs.mdx.jp/ja/main/faq.html#ip)に従って割り当て可能なIPアドレス範囲を確認し、範囲内かつ未使用のIPアドレス範囲からmaster_ip_addressを設定してください。

In [ ]:
if master_provider != "onpremises":
    print(vcp.get_vpn_catalog(master_provider).get('private_network_ipmask'))

次のセルに、マスターノードに割り当てるIPアドレスを指定してください。上のセルの出力結果に表示された範囲のIPアドレスを指定してください。

In [ ]:
# (例)
# master_ipaddress = '172.30.2.120'

master_ipaddress = 

マスターノードに設定するホスト名を指定してください。

In [ ]:
# (例)
# master_hostname = 'master'

master_hostname = 'master'

### マスターノードのmdxパック数

mdx上で構築する場合、マスターノードのパック数を指定します。mdx以外の環境では、この節をスキップしてください。

In [ ]:
# (例)
# mdx_master_pack_num = 4

mdx_master_pack_num = 4

### パラメータの保存

この章で指定したパラメータの値をファイルに保存します。

値の保存を行う前に、入力されたパラメータに対して簡易なチェックを行います。エラーになった場合はその後に表示される指示に従ってください。

In [ ]:
%run scripts/utils.py
check_parameters_with_keys(
    globals(),
    [
        "master_provider",
        "master_flavor",
        "master_root_size",
        "master_ipaddress",
        "master_hostname",
    ],
    ["vcp"],
)

この章で指定したパラメータを `group_vars` ファイルに保存します。

In [ ]:
%run scripts/group.py
update_group_vars(
    ugroup_name,
    master_flavor=master_flavor,
    master_root_size=master_root_size,
    master_provider=master_provider,
    master_ipaddress=master_ipaddress,
    master_hostname=master_hostname,
)

if 'master_instance_type' in vars():
    update_group_vars(
        ugroup_name, 
        master_instance_type=master_instance_type,
    )

if 'mdx_master_pack_num' in vars():
    update_group_vars(
        ugroup_name,
        mdx_master_pack_num=mdx_master_pack_num,
    )

`group_vars`ファイルの内容を確認してみます。

In [ ]:
!cat group_vars/{ugroup_name}.yml

## NFS用ディスク

NFS用ディスクに割り当てるリソースを指定します。

> NFS用に仮想ディスクを作成しない構成にする場合は、この章をスキップしてください。

mdx上でOpenHPC環境を構築する場合、VCディスクは非対応のためこの章をスキップしてください。

![NFS](images/ohpc-004.png)

ディスクサイズ(GB)を指定してください。16GB 以上の値を指定してください。

In [ ]:
# (例)
# nfs_disk_size = 64

nfs_disk_size = 

ディスクのデバイス名を指定してください。デバイス名はプロバイダやインスタンスタイプによって異なる値となります。

In [ ]:
# (例)
# nfs_device = '/dev/nvme1n1' # AWS
# nfs_device = '/dev/xvdf'    # AWS(Xenインスタンスタイプ)
# nfs_device = '/dev/sdc'     # Azure
# nfs_device = '/dev/sdb'     # Oracle Cloud

nfs_device = 

NFS用に仮想ディスクを作成しない構成の場合、誤ってディスクに関するパラメータを設定してしまった場合は、次のセルのコメントを外して実行してください。

In [ ]:
# del(nfs_disk_size)
# del(nfs_device)

### パラメータの保存

この章で指定したパラメータの値をファイルに保存します。

値の保存を行う前に、入力されたパラメータに対して簡易なチェックを行います。エラーになった場合はその後に表示される指示に従ってください。

In [ ]:
%run scripts/utils.py
if 'nfs_disk_size' in vars():
    check_parameters_with_keys(globals(), ["nfs_disk_size", "nfs_device"], ["master_provider"])

この章で指定したパラメータを `group_vars` ファイルに保存します。

In [ ]:
%run scripts/group.py
if 'nfs_disk_size' in vars():
    update_group_vars(
        ugroup_name,
        nfs_disk_size=nfs_disk_size,
        nfs_device=nfs_device,
    )


`group_vars`ファイルの内容を確認してみます。

In [ ]:
!cat group_vars/{ugroup_name}.yml

## 計算ノードのリソース設定

計算ノードの物理的なリソース（CPU、メモリ、GPU等）に関するパラメータを指定します。

![計算ノード](images/ohpc-002.png)

### 計算ノードのクラウドプロバイダ

VCノードを作成するプロバイダ名を指定します。

In [ ]:
# (例)
# compute_provider = 'aws'
# compute_provider = 'azure'
# compute_provider = 'oracle'
# compute_provider = 'onpremises'   ### mdxの場合

compute_provider = 

### 計算ノードのflavor

計算ノードに割り当てるリソースに対応する `flavor` の値を指定します。

VCノードの`spec`に対して種々のパラメータを毎回設定するのは煩雑な作業になります。そこでVCP SDKでは典型的なパラメータセットを事前に定義しています。事前に定義したパラメータセットのことをVCP SDKでは`flavor`と呼んでいます。`spec`に設定できるパラメータはクラウドプロバイダ毎に異なるので `flavor`もプロバイダ毎に定義されています。

次のセルを実行すると`compute_provider`で指定したクラウドプロバイダ名に対応する `flavor` の一覧が表示されます。

In [ ]:
vcp.df_flavors(compute_provider)

上に表示された表の `flavor` の欄の値から、計算ノードとして利用するVCノードの `flavor` を選んで次のセルで指定してください。

In [ ]:
# (例)
# compute_flavor = 'medium'
# compute_flavor = 'gpu'
# compute_flavor = 'default'    ### mdxの場合

compute_flavor = 

### 計算ノードのインスタンスタイプ

`flavor`で事前に定義されている以外のインスタンスタイプを利用したい場合は次のセルのコメントを外してインスタンスタイプを指定してください。

In [ ]:
# (例)
# compute_instance_type = 'g5.xlarge'              # AWS NVIDIA A10G
# compute_instance_type = 'Standard_NC4as_T4_v3'   # Azure NVIDIA T4
# compute_instance_type = 'VM.GPU2.1'              # Oracle Cloud  NVIDIA P100

### 計算ノードのルートボリュームサイズ

計算ノードのルートボリュームサイズを変更する必要がある場合は、次のセルのコメントを外してサイズ(GB)を指定してください。

In [ ]:
# (例)
# compute_root_size = 16

### 計算ノードにおけるGPUの利用

計算ノードでGPUを利用するか否かについて指定してください。

In [ ]:
# (例)
# compute_use_gpu = False  # GPU を利用しない
# compute_use_gpu = True   # GPU を利用する

compute_use_gpu = False

`compute_use_gpu`を`True`に設定することで、計算ノードを起動する際のVMイメージ、BaseコンテナイメージとしてNVIDIAドライバがセットアップされているものが選択されます。ただし`compute_flavor`や`compute_instance_type`で指定したインスタンスタイプがGPUを利用可能なものである必要があります。

GPUを利用する場合は１ノードあたりのGPU数を次のセルで指定してください。指定する値は`compute_flavor`または`compute_instance_type`で指定した計算ノードのインスタンスタイプが保持するGPU数と一致する値を指定してください。指定しない場合（かつ`compute_use_gpu = True`）は `compute_gpus = 1`とみなします。

In [ ]:
# (例)
# compute_gpus = 1

### 計算ノードのノード数

計算ノードとして作成するノード数を指定します。

In [ ]:
# (例)
# compute_nodes = 4

compute_nodes = 

初回構築時の計算ノード数は上のセルで指定した`compute_nodes`の値となります。

> 構築後に「081-計算ノードの追加.ipynb」「082-計算ノードの削除.ipynb」を実行することにより計算ノード数を変更することができます。

### 計算ノードのmdxパック数

mdx上で構築する場合、計算ノードのパック数を指定します。mdx以外の環境では、この節をスキップしてください。

In [ ]:
# (例)
# mdx_compute_pack_num = 4

mdx_compute_pack_num = 4

### チェック

この章で指定したパラメータが適切かどうかチェックします。エラーになった場合はその後に表示される指示に従ってください。

In [ ]:
%run scripts/utils.py
check_parameters_with_keys(
    globals(),
    [
        "compute_provider",
        "compute_flavor",
        "compute_root_size",
        "compute_use_gpu",
        "compute_gpus",
        "compute_nodes",
    ],
    ["vcp"],
)

## 計算ノードのSlurm設定

[Slurm](https://slurm.schedmd.com/archive/slurm-24.11.5/quickstart.html)のFeature、Partition設定を行います。これらはジョブスケジューリングとリソース管理の基盤となります。

### Slurmの概念

Slurmは**Feature**、**Nodeset**、**Partition**という概念でリソースを管理します。

**Feature**は計算ノードに付与する属性タグで、「GPU搭載」「大容量メモリ」といったハードウェア特性を表します。ジョブ実行時のノード選択条件として使われます。

**Nodeset**は同じ特性を持つノードの集合です。

**Partition**はジョブを投入する論理的な区画で、複数のNodesetを含めたり、アクセス制御を設定したりできます。

### テンプレートの設計方針

このテンプレートでは、Feature名をVCP SDKの**VC Unit名**として使用します。指定したFeature名を起点として、NodesetとPartitionを構成します。

```
Feature名（VC Unit名）を指定
  ↓ Nodeset名を自動生成: ns-{feature_name}
  ↓ Partitionに割り当て
Partition（ジョブ投入先）
```

NodesetはSlurmの**Dynamic Nodes**機能により自動生成されるため、ユーザーが手動で定義する必要はありません。

**具体例**: Feature名 `gpu-a100-lab1` を指定した場合

- Nodeset名: `ns-gpu-a100-lab1`（自動生成）
- Partition `gpu` にこのNodesetを割り当て → ユーザーはジョブを `gpu` パーティションに投入

複数のFeatureを1つのPartitionに統合することも可能です。たとえばAWSとAzureに別々にGPUノードを配備した場合、両方のNodesetを同一のPartitionに割り当てることで、どちらのノードでもジョブを実行できます。

### Feature名の指定

計算ノードの識別子となるFeature名を指定します。

**Feature名とは**:

- 計算ノードのハードウェア特性と利用環境を一意に識別する名前です
- VCP SDKの **VC Unit名** として使用されます
- Slurmのノード管理の基盤となる識別子です

**命名ガイドライン**（`{hardware}-{environment}` 形式を推奨）:

- 英小文字、数字、ハイフンのみ使用可能（1–64文字）

**命名例**:

| feature_name | ハードウェア | 環境 |
|---|---|---|
| `cpu-aws` | 汎用CPU | AWS |
| `gpu-a100-lab1` | NVIDIA A100 | 研究室1 |
| `gpu-a100-lab2` | NVIDIA A100 | 研究室2 |
| `highmem-azure` | 大メモリ | Azure |
| `cpu-onpremises` | 汎用CPU | mdx/オンプレ |

**パーティションとの関係**:

Feature名は、次節で設定するPartitionのNodeset名 `ns-{feature_name}` として参照されます。
例: `feature_name = "gpu-a100-lab1"` → Nodeset名: `ns-gpu-a100-lab1`

> 複数のFeatureを追加する場合は、「083-Featureの追加.ipynb」を使用します。
> 同一ハードウェアで複数Featureを作る場合は環境識別子（`lab1`, `lab2` など）で区別してください。

In [ ]:
# (例)
# feature_name = 'cpu-aws'
# feature_name = 'gpu-a100-lab1'
# feature_name = 'highmem-azure'
# feature_name = 'cpu-onpremises'   ### mdxの場合

feature_name = 

指定されたFeature名は、Slurmの設定とVCP SDKのリソース管理の両方で使用されます。

**Feature名とVCユニット名の関係**:

- Feature名は、VCP SDKの**Unit名**として使用されます
- 例: Feature名を `gpu-a100-aws` とした場合、VCP上では以下の構成になります
  - `UnitGroup: {ugroup_name}`
    - `Unit: master` (マスターノード)
    - `Unit: gpu-a100-aws` (計算ノード)


### Partition名

このテンプレートでは初期構築として1つのSlurmパーティションを作成します。Slurmのパーティション名を指定します。

**例**:

- 汎用的な計算ノード: `compute`
- GPU専用ノード: `gpu`
- 用途別: `analysis`, `simulation` など

> 複数のパーティションが必要な場合は、初期構築完了後に「083-Featureの追加.ipynb」を使用して追加できます。

指定したパーティション名は、ジョブ投入先の論理区画として使用されます。

In [ ]:
# (例)
# partition_name = 'compute'
# partition_name = 'gpu'

partition_name = 

### アクセス制御設定（オプション）

Partitionへのアクセスを特定のグループに制限したい場合は、この節で設定してください。この設定はオプションです。アクセス制限が不要な場合は、この節をスキップしてください。

**アクセス制御とは**:

SlurmのPartitionには、`AllowGroups`パラメータを使用してアクセスできるUNIXグループを制限できます。これにより、特定のユーザーグループのみが特定のPartitionにジョブを投入できるようになります。

**使用例**:

- GPU Partitionを特定の研究グループに制限
  ```python
  partition_allow_groups = ["gpu-users", "research-team"]
  ```
- 管理者のみがアクセス可能なテストPartition
  ```python
  partition_allow_groups = ["admin"]
  ```
- 全ユーザーがアクセス可能（デフォルト）
  ```python
  partition_allow_groups = []  # 空リストまたは未定義
  ```

**注意事項**:

- 指定するグループ名は、計算ノードおよびマスターノードのUNIXシステムに存在する必要があります
- グループ管理は別途Ansible等で実施する必要があります
- 空リスト `[]` または未定義の場合は、アクセス制限なし（全ユーザーがアクセス可能）

In [ ]:
# (例)
# partition_allow_groups = ["all-users"]           # all-usersグループのみ
# partition_allow_groups = ["gpu-users", "admin"]  # gpu-usersとadminグループ
# partition_allow_groups = []                      # 制限なし（全ユーザー）

# partition_allow_groups = []

### チェック

この章で指定したパラメータが適切かどうかチェックします。エラーになった場合はその後に表示される指示に従ってください。

In [ ]:
%run scripts/utils.py
check_parameters_with_keys(
    globals(),
    [
        "feature_name",
        "partition_name",
        "partition_allow_groups",
    ],
    [],
)

## 計算ノードのネットワーク設定

計算ノードのIPアドレス、ホスト名、NFSマウントポイントを設定します。

### 計算ノードのIPアドレス

計算ノードに割り当てるIPアドレスレンジを指定します。

**IPレンジの指定方法**:

IPアドレスレンジは以下の形式で指定します:
- 短縮形式: `"172.30.2.121-130"` （第4オクテットのみ指定）
- 完全形式: `"172.30.2.121-172.30.2.130"` （開始と終了の完全なIPアドレス）

**複数レンジの指定**:

非連続なIPアドレスを使用する場合は、リストで複数のレンジを指定できます:
```python
compute_ip_ranges = ["172.30.2.121-130", "172.30.2.151-160"]
```

この例では、121-130と151-160の20個のIPアドレスが使用可能になります。

**IPアドレス数と compute_nodes の関係**:

- IPレンジから計算される総IP数は、`compute_nodes` 以上である必要があります
- 初回構築時は `compute_nodes` 個のノードが起動しますが、将来IPレンジで指定したノード数まで増やせます

**注意事項**:

- 他のノード（マスターノードなど）と重複しないアドレスを指定してください
- パーティションを追加する場合、他パーティションと重複しないように計画してください

VCノードに割り当て可能なIPアドレスの範囲を確認します。

> mdx上での構築の場合には、以下のセルではアドレス範囲を取得できないため、[mdxでの静的IPアドレス設定方法](https://docs.mdx.jp/ja/main/faq.html#ip)に従って割り当て可能なIPアドレス範囲を確認し、範囲内かつ未使用のIPアドレス範囲からcompute_ip_rangesを設定してください。

In [ ]:
if compute_provider != "onpremises":
    print(vcp.get_vpn_catalog(compute_provider).get('private_network_ipmask'))

計算ノードに割り当てるIPアドレスレンジを指定します。

In [ ]:
# (例)
# compute_ip_ranges = ["172.30.2.121-130"]                            # 10個のIPアドレス (121-130)
# compute_ip_ranges = ["172.30.2.121-172.30.2.130"]                   # 10個のIPアドレス (121-130)
# compute_ip_ranges = ["172.30.2.121-130", '172.30.2.151-160']        # 20個のIPアドレス (非連続)
# compute_ip_ranges = [                                               # 30個のIPアドレス
#     "172.30.2.121-130",
#     "172.30.2.151-160",
#     "172.30.2.201-210",
# ]

compute_ip_ranges = [
    
]

### 計算ノードのホスト名テンプレート

計算ノードのホスト名テンプレートを指定します。

**ホスト名テンプレートとは**:

ホスト名にはPartitionやFeatureに応じたプレフィックスにノードの通番を付与する構成とします。例えばプレフィックスを `c` とした場合、ホスト名（非修飾ホスト名）は `c1`, `c2`, `c3`, ... のようになります。

通番の部分を動的に生成するので、ホスト名のテンプレートをPythonの[書式指定文字列](https://docs.python.org/ja/3.13/library/string.html#formatspec)により指定してください。

**例**:

- `"c{}"` → c1, c2, c3, ...
- `"c{:02}"` → c01, c02, c03, ...
- `"gpu{}"` → gpu1, gpu2, gpu3, ...
- `"gpu{:02}"` → gpu01, gpu02, gpu03, ...

**Partitionとの関連**:

ホスト名のプレフィックスは、通常Partition名やFeatureの性質に合わせて命名します:
- GPU Partition → `gpu{}`
- CPU Partition → `c{}`
- 高メモリPartition → `mem{}`

In [ ]:
# (例)
# compute_hostname_template = "c{}"       # c1, c2, c3 ...
# compute_hostname_template = "c{:02}"    # c01, c02, c03, ...
# compute_hostname_template = "gpu{:02}"  # gpu01, gpu02, gpu03, ...

compute_hostname_template = 

### 計算ノードのNFSマウントポイント

計算ノードに追加のNFSマウントポイントを設定したい場合は、この節でNFSエントリを指定してください。

> この設定はオプションです。追加のNFSマウントポイントが不要な場合は、この節をスキップしてください。

計算ノードには、デフォルトで以下の2つのNFSマウントポイントが設定されます：

* `/opt/ohpc/pub`
  - マスターノードの `/ohpc/pub` をマウント（読み取り専用）
* `/home`
  - マスターノードの `/home` をマウント（読み書き可能）

これらに加えて他のNFSサーバーからマウントする必要がある場合、以下のセルの`compute_optional_nfs_entries`でNFSエントリを追加することができます。

入力形式は`/etc/fstab`の書式に従います。fstabの記述例を以下に示します。参考にしてセルを設定して下さい。

```
server1.example.com:/data /mnt/nfs/data nfs nfsvers=4,rw,nodev 0 0
192.168.1.100:/shared /mnt/shared nfs nfsvers=4,rw 0 0
```

`compute_optional_nfs_entries`は複数行の文字列として指定してください。

In [ ]:
compute_optional_nfs_entries = """

"""

### チェック

この章で指定したパラメータが適切かどうかチェックします。エラーになった場合はその後に表示される指示に従ってください。

In [ ]:
%run scripts/utils.py
check_parameters_with_keys(
    globals(),
    [
        "compute_ip_ranges",
        "compute_hostname_template",
        "compute_optional_nfs_entries",
    ],
    [
        "vcp",
        "master_ipaddress",
        "compute_provider",
        "compute_flavor",
        "compute_nodes",
    ],
)

## 計算ノードのパラメータ保存

ここまでの計算ノードに関する「計算ノードのリソース設定」「計算ノードのSlurm設定」「計算ノードのネットワーク設定」で指定したパラメータの値をファイルに保存します。

パラメータを `group_vars` ファイルに保存します。

In [ ]:
%run scripts/utils.py
%run scripts/group.py

feature_name, feature_config = generate_slurm_features_params(vars())
update_slurm_features(ugroup_name, feature_name, feature_config)
nodeset_name = f"ns-{feature_name}"

partition_config = {"nodesets": [nodeset_name], "default": True}
# アクセス制御設定を追加（指定されている場合）
if 'partition_allow_groups' in vars() and partition_allow_groups:
    partition_config["allow_groups"] = partition_allow_groups

update_slurm_partitions(ugroup_name, partition_name, partition_config)

`group_vars`ファイルの内容を確認してみます。

In [ ]:
!cat group_vars/{ugroup_name}.yml

## CoreDNS設定

クラスタ内部の名前解決を提供するCoreDNSに関するパラメータを指定します。

![CoreDNS](images/ohpc-005.png)

### CoreDNSとは

**CoreDNS**は、クラスタ内部のホスト名解決を提供する軽量なDNSサーバーです。OpenHPC-v3環境では以下の役割を果たします:

* **動的ノードの名前解決**: 動的に追加・削除される計算ノードのホスト名をマスターノードが管理
* **パーティション間の通信**: 複数パーティション構成で、異なるパーティションのノード間でホスト名による通信が可能
* **設定の自動更新**: 計算ノードの追加・削除時に自動的にDNSレコードが更新される（5秒間隔）

**動作の仕組み**:

1. マスターノードでCoreDNSが起動し、クラスタ内部のDNSサーバーとして機能
2. 計算ノードは起動時にマスターノードのIPアドレスをDNSサーバーとして設定
3. CoreDNSは `/opt/ohpc/pub/etc/hosts.ohpc` ファイルを参照してホスト名を解決
4. Feature管理Notebookが `/opt/ohpc/pub/etc/hosts.ohpc` を自動更新

### DNSフォワーダー

クラスタ外部のホスト名を解決する際に使用するDNSフォワーダーを指定します。

CoreDNSがクラスタ内ホスト以外の名前解決要求を受けた場合、ここで指定したDNSサーバーに問い合わせを転送します。

**設定例**:
- Google Public DNS: `8.8.8.8`, `8.8.4.4`
- Cloudflare DNS: `1.1.1.1`, `1.0.0.1`
- 組織内DNSサーバー: `192.168.1.1`

リスト形式で複数指定できます。デフォルトではGoogle Public DNSを使用します。

In [ ]:
# (例)
# dns_forwarders = ['8.8.8.8', '8.8.4.4']              # Google Public DNS
# dns_forwarders = ['1.1.1.1', '1.0.0.1']              # Cloudflare DNS
# dns_forwarders = ['192.168.1.1']                     # 組織内DNSサーバー
# dns_forwarders = ['192.168.1.1', '8.8.8.8']          # 組織内DNS + フォールバック

dns_forwarders = ['8.8.8.8', '8.8.4.4']

### クラスタ内部ドメイン名

クラスタ内部で使用するドメイン名を指定します。

計算ノードのFQDN（完全修飾ドメイン名）は `{hostname}.{dns_domain}` の形式になります。例えば:
- dns_domain = "ohpc.internal"
- hostname = "c1"
- FQDN = "c1.ohpc.internal"

ドメイン名のデフォルト値は `ohpc.internal`となります。デフォルト値以外を指定する場合は、次のセルのコメントを外しドメイン名を指定してください。

In [ ]:
# (例)
# dns_domain = 'ohpc.internal'

### パラメータの保存

この章で指定したパラメータの値をファイルに保存します。

値の保存を行う前に、入力されたパラメータに対して簡易なチェックを行います。エラーになった場合はその後に表示される指示に従ってください。

In [ ]:
%run scripts/utils.py
check_parameters_with_keys(
    globals(),
    ["dns_forwarders", "dns_domain"],
    [],
)

この章で指定したパラメータを `group_vars` ファイルに保存します。

In [ ]:
%run scripts/group.py

update_group_vars(
    ugroup_name,
    dns_forwarders=dns_forwarders,
    dns_domain=dns_domain if "dns_domain" in vars() else "ohpc.internal"
)

`group_vars`ファイルの内容を確認してみます。

In [ ]:
!cat group_vars/{ugroup_name}.yml

## mdx共通パラメータ

mdx上でOpenHPC環境を構築する場合に必要な共通パラメータを設定します。
mdx以外の環境では、この章をスキップしてください。

### mdx REST APIの準備

mdx REST APIを利用するための準備をします。

以下のセルを実行してmdx REST API認証トークンを入力します。

In [ ]:
from getpass import getpass
mdx_token = getpass("mdx API token")

mdx REST APIエンドポイントにIPv6で接続しようとすると到達不可となる場合があるため、以下のセルを実行してIPv4での接続を強制します。

In [ ]:
%run scripts/mdx_ops.py
use_ipv4_only()

VCP SDK mdx用プラグインモジュールを読み込みます。

In [ ]:
from common import logsetting
from vcpsdk.plugins.mdx_ext import MdxResourceExt
mdx = MdxResourceExt(mdx_token)

利用可能なmdxのプロジェクト情報を確認します。

In [ ]:
import json
projects = mdx.get_assigned_projects()
print(json.dumps(projects[0]["projects"], indent=2))

OpenHPC環境を構築するプロジェクト名を、mdx_project_nameに設定します。

In [ ]:
mdx_project_name = 'プロジェクト名'

今後のVMセットアップのため、操作対象のプロジェクトを設定します。

以下のセルがエラーとなる場合はプロジェクト名の指定に誤りがありますので、上のセルのプロジェクト名設定を確認してください。

In [ ]:
mdx.set_current_project_by_name(mdx_project_name)

プロジェクトで利用可能なネットワークセグメントのリストを取得し、先頭のIDを設定します。

In [ ]:
segments = mdx.get_segments()
print(json.dumps(segments, indent=2))

mdx_segment_id = mdx.get_segments()[0]["uuid"]
print(mdx_segment_id)

### パラメータの保存

この章で指定したパラメータの値をファイルに保存します。

値の保存を行う前に、入力されたパラメータに対して簡易なチェックを行います。エラーになった場合はその後に表示される指示に従ってください。

In [ ]:
%run scripts/utils.py

check_parameters(
    _params=dict(
        vcp=vcp, 
        master_provider=master_provider,
        compute_provider=compute_provider,
    ),
    mdx_project_name=mdx_project_name,
    mdx_segment_id=mdx_segment_id,
)

この章で指定したパラメータを group_vars ファイルに保存します。

In [ ]:
%run scripts/group.py

update_group_vars(
    ugroup_name,
    mdx_ssh_user_name="mdxuser",
    mdx_project_name=mdx_project_name,
    mdx_segment_id=mdx_segment_id,
)

`group_vars`ファイルの内容を確認してみます。

In [ ]:
!cat group_vars/{ugroup_name}.yml

## Slurm

Slurmに関連するパラメータを指定します。

![etc_hosts](images/ohpc-006.png)

### munge.key

[MUNGE](https://dun.github.io/munge/)はHPCクラスタ環境のための認証サービスです。この節ではSLURMがコンポーネント間の認証に利用するMUNGEの鍵ファイル`munge.key`を作成します。

`munge.key` に書き込む内容を乱数から生成します。

In [ ]:
import secrets

munge_key = secrets.token_bytes(1024)

`munge.key` の内容は秘匿情報になるので、`group_vars`ファイルではなく VC Controller の HashiCorp Vault に保存します。
HashiCorp Vault は秘密情報を保存するための Key Valueストアです。保持する情報は暗号化されます。


HashiCorp Valutのなかの記録場所となるパスを次のセルで指定します。

In [ ]:
vault_path_munge_key = f'cubbyhole/OpenHPC/{ugroup_name}/munge.key'
print(vault_path_munge_key)

### パラメータの保存

この章で指定したパラメータの値をファイルに保存します。`munge.key`は秘匿情報のため、暗号化され記録される HashiCorp Vaultに保存します。他のパラメータについては `group_vars`ファイルに保存します。

`munge.key`の内容をVCCのHashiCorp Valutに保存します。次のセルを実行してください。

> 保存に成功すると `<Response [204]>` と表示されます。

In [ ]:
import requests
import base64
%run scripts/group.py
gvars = load_group_vars(ugroup_name)
payload = {
    'munge.key': base64.b64encode(munge_key).decode('UTF-8'),
}

vault_url = f'{vcp.vcc_info()["vault_url"]}/v1/{vault_path_munge_key}'

custom_headers = {
    'X-Vault-Token': vcc_access_token,
}

r = requests.post(
        vault_url,
        headers=custom_headers,
        json=payload, 
        verify=gvars.get("verify_ssl_certificate", True),
    )
r

他の値を `group_vars`ファイルに保存します。

In [ ]:
%run scripts/group.py

update_group_vars(
    ugroup_name,
    vault_path_munge_key=vault_path_munge_key,
)

`group_vars`ファイルの内容を確認してみます。

In [ ]:
!cat group_vars/{ugroup_name}.yml

## チェック

設定項目漏れがないことを確認します。

次のセルを実行しエラーとならないことを確認してください。エラーになった場合は、このNotebookの中に実行していないセルがないかを確認してください。

In [ ]:
%run scripts/group.py
gvars = load_group_vars(ugroup_name)
require_params = [
    'master_flavor', 'master_hostname', 'master_ipaddress', 'master_root_size',
    'master_provider', 'slurm_features', 'slurm_partitions',
    'ugroup_name', 'ssh_private_key_path', 'ssh_public_key_path',
    'vault_path_munge_key', 'dns_forwarders', 'dns_domain',
]

for x in require_params:
    if x not in gvars:
        raise RuntimeError("ERROR: not set {}".format(x))